In [36]:
#Import Libraries
import pandas as pd
import numpy as np
import re
import string

In [37]:
#Load Merged Dataset
df = pd.read_csv("../data/processed/merged_news.csv")
print(f"Total articles: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(5))

Total articles: 10790
Columns: ['title', 'url', 'date', 'category', 'source']
                                               title  \
0  Komisi X Akan Minta Penjelasan Kemendikdasmen ...   
1  Full Senyum, Prabowo-Megawati Gandengan Tangan...   
2  Purbaya soal Ekspor via PT DSI: Bukan Program ...   
3  Prabowo: Ekonomi RI Tumbuh tapi Apakah Sudah D...   
4  Prabowo: Tak Ada Bangsa yang Kasihan Sama Kita...   

                                                 url         date category  \
0  https://nasional.kompas.com/read/2026/06/01/11...  1 Juni 2026     News   
1  https://nasional.kompas.com/read/2026/06/01/11...  1 Juni 2026     News   
2  https://money.kompas.com/read/2026/06/01/11155...  1 Juni 2026    Money   
3  https://nasional.kompas.com/read/2026/06/01/11...  1 Juni 2026     News   
4  https://nasional.kompas.com/read/2026/06/01/10...  1 Juni 2026     News   

   source  
0  Kompas  
1  Kompas  
2  Kompas  
3  Kompas  
4  Kompas  


In [38]:
# Data Quality Check
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Date Format Sample ===")
print(df["date"].value_counts().head(10))

print("\n=== Source Distribution ===")
print(df["source"].value_counts())

print("\n=== Duplicates ===")
print(f"Duplicate titles: {df.duplicated(subset='title').sum()}")
print(f"Duplicate URLs: {df.duplicated(subset='url').sum()}")

=== Missing Values ===
title          0
url            0
date         176
category    6260
source         0
dtype: int64

=== Date Format Sample ===
date
20 Mei 2026         125
2025/05/20          117
13 Februari 2026     89
2025/08/15           87
1 Mei 2026           81
16 Mei 2026          75
2 Februari 2026      73
22 Januari 2026      73
4 Februari 2026      70
23 Januari 2026      68
Name: count, dtype: int64

=== Source Distribution ===
source
Tempo     6260
Kompas    4530
Name: count, dtype: int64

=== Duplicates ===
Duplicate titles: 524
Duplicate URLs: 437


In [39]:
#cleam title
def clean_title(text):
    if not isinstance(text, str):
        return None
    # Remove newlines and extra whitespace
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    # Remove category and date patterns that got mixed in
    text = re.sub(r'\b(News|Money|Nasional|Edukasi|Cahaya|Tekno|Bola|Entertainment)\b.*', '', text)
    text = re.sub(r'\d+ \w+ 20\d\d.*', '', text)
    return text.strip()

df["title"] = df["title"].apply(clean_title)

# Preview
print(df["title"].head(10))

0    Komisi X Akan Minta Penjelasan Kemendikdasmen ...
1    Full Senyum, Prabowo-Megawati Gandengan Tangan...
2    Purbaya soal Ekspor via PT DSI: Bukan Program ...
3    Prabowo: Ekonomi RI Tumbuh tapi Apakah Sudah D...
4    Prabowo: Tak Ada Bangsa yang Kasihan Sama Kita...
5    Daftar Mantan Presiden-Wapres Hadiri Upacara H...
6    Momen Prabowo Berdoa di Depan Peti Jenazah Rya...
7    Meski Dipersilakan Prabowo, Megawati Tolak Ber...
8    Didampingi Gibran, Prabowo Pimpin Upacara Hari...
9    Romo Syafi’i: Gagasan Prabowo Bentuk Kemenhaj ...
Name: title, dtype: str


In [40]:
mask = df["title"].str.match(r"^\d{2}:\d{2}")
print(f"Articles with timestamp prefix: {mask.sum()}")
print(df[mask]["title"].head(10))

Articles with timestamp prefix: 234
3214    07:53 Fase Dua Gencatan Senjata di Gaza: TNI "...
3220    04:41 Angka Kenaikan UMP Dianggap Tak Cukup un...
3227    03:58 Thomas Pastikan Tak Ada yang Dilanggar D...
3229    03:03 Sudah Gugat UMP ke PTUN, Buruh Akan Meng...
3230    03:18 Demo Buruh Ditutup dengan Nyalakan Flare...
3249    03:30 Prabowo Kumpulkan Menteri di Hambalang S...
3252    03:44 Prabowo Ikut Iuran Anggota Dewan Perdama...
3264    08:55 Tok! DPR Setujui Polri Tetap di Bawah Pr...
3314    04:13 Prabowo Panggil Purbaya hingga Bahlil ke...
3330    01:31 Menkes: Cek Kesehatan Gratis adalah Mimp...
Name: title, dtype: str


In [41]:
#Remove timestamp prefix from titles
df["title"] = df["title"].str.replace(r"^\d{2}:\d{2}\s*", "", regex=True)

# Verify
mask = df["title"].str.match(r"^\d{2}:\d{2}")
print(f"Remaining timestamp prefix: {mask.sum()}")
print(df["title"].head(5))

Remaining timestamp prefix: 0
0    Komisi X Akan Minta Penjelasan Kemendikdasmen ...
1    Full Senyum, Prabowo-Megawati Gandengan Tangan...
2    Purbaya soal Ekspor via PT DSI: Bukan Program ...
3    Prabowo: Ekonomi RI Tumbuh tapi Apakah Sudah D...
4    Prabowo: Tak Ada Bangsa yang Kasihan Sama Kita...
Name: title, dtype: str


In [42]:
# Standardize Date Format
months_map = {
    "Januari": "01", "Februari": "02", "Maret": "03", "April": "04",
    "Mei": "05", "Juni": "06", "Juli": "07", "Agustus": "08",
    "September": "09", "Oktober": "10", "November": "11", "Desember": "12"
}

def standardize_date(date_str):
    if not isinstance(date_str, str):
        return None
    if re.match(r"\d{4}/\d{2}/\d{2}", date_str):
        return date_str.replace("/", "-")
    parts = date_str.strip().split(" ")
    if len(parts) == 3:
        day, month, year = parts
        month_num = months_map.get(month)
        if month_num:
            return f"{year}-{month_num}-{day.zfill(2)}"
    return None

df["date"] = df["date"].apply(standardize_date)
df["date"] = pd.to_datetime(df["date"])

print(f"Date range: {df['date'].min()} → {df['date'].max()}")
print(df.isnull().sum())

Date range: 2024-10-01 00:00:00 → 2026-06-01 00:00:00
title          0
url            0
date         176
category    6260
source         0
dtype: int64


In [43]:
#Filter from October 2024
df = df[df["date"] >= "2024-10-01"]
print(f"Articles after filter: {len(df)}")
print(f"Date range: {df['date'].min()} -> {df['date'].max()}")

Articles after filter: 10614
Date range: 2024-10-01 00:00:00 -> 2026-06-01 00:00:00


In [44]:
#Check date range per source
print("Kompas date range:")
print(df[df["source"]=="Kompas"]["date"].min(), "->", df[df["source"]=="Kompas"]["date"].max())

print("\nTempo date range:")
print(df[df["source"]=="Tempo"]["date"].min(), "->", df[df["source"]=="Tempo"]["date"].max())

Kompas date range:
2025-12-15 00:00:00 -> 2026-06-01 00:00:00

Tempo date range:
2024-10-01 00:00:00 -> 2026-06-01 00:00:00


In [45]:
#Check if duplicate titles are cross-source
dup_titles = df[df.duplicated(subset="title", keep=False)]
cross_source = dup_titles.groupby("title")["source"].nunique()
print(f"Duplicate titles from SAME source: {(cross_source == 1).sum()}")
print(f"Duplicate titles from DIFFERENT source: {(cross_source == 2).sum()}")


Duplicate titles from SAME source: 220
Duplicate titles from DIFFERENT source: 1


In [46]:
#Show actual duplicate titles with both sources
dup_titles = df[df.duplicated(subset="title", keep=False)]
dup_grouped = dup_titles.sort_values("title")
print(dup_grouped[["title", "url", "source"]].head(20).to_string())

                                                           title                                                                                                                                       url source
9214                                                              https://www.tempo.co/politik/nasional-sepekan-sejumlah-menteri-temui-jokowi-hingga-intimidasi-ke-penulis-esai-konspirasi-prabowo-1230569  Tempo
9232                                                              https://www.tempo.co/politik/nasional-sepekan-sejumlah-menteri-temui-jokowi-hingga-intimidasi-ke-penulis-esai-konspirasi-prabowo-1230569  Tempo
5966                                                                                       https://www.tempo.co/politik/nasional-sepekan-jonan-dipanggil-prabowo-hingga-ledakan-di-sman-72-jakarta-2087791  Tempo
8830  27 Tahun Reformasi: Tim Mawar Pion Penculikan Aktivis 1998                                            https://www.tempo.co/politik/27-tahun-reformasi-tim-

In [47]:
# CELL 6 — Drop duplicates by URL
df = df.drop_duplicates(subset="url", keep="first")

In [48]:
#Drop rows with missing date
df = df.dropna(subset=["date"])

print(f"Total after cleaning: {len(df)}")
print(f"\nDate range: {df['date'].min()} -> {df['date'].max()}")
print(f"\n{df['source'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Total after cleaning: 10198

Date range: 2024-10-01 00:00:00 -> 2026-06-01 00:00:00

source
Tempo     5673
Kompas    4525
Name: count, dtype: int64

Missing values:
title          0
url            0
date           0
category    5673
source         0
dtype: int64


In [49]:
#Drop category column
df = df.drop(columns=["category"])
print(df.columns.tolist())
print(df.isnull().sum())

['title', 'url', 'date', 'source']
title     0
url       0
date      0
source    0
dtype: int64


In [50]:
#Drop duplicate titles, keep first
print(f"Before: {len(df)}")
df = df.drop_duplicates(subset="title", keep="first")
print(f"After: {len(df)}")
print(df["source"].value_counts())

Before: 10198
After: 10069
source
Tempo     5553
Kompas    4516
Name: count, dtype: int64


In [51]:
#Final Check
print(f"Total articles: {len(df)}")
print(f"\nSource distribution:\n{df['source'].value_counts()}")
print(f"\nDate range: {df['date'].min()} → {df['date'].max()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nColumns: {df.columns.tolist()}")

Total articles: 10069

Source distribution:
source
Tempo     5553
Kompas    4516
Name: count, dtype: int64

Date range: 2024-10-01 00:00:00 → 2026-06-01 00:00:00

Missing values:
title     0
url       0
date      0
source    0
dtype: int64

Columns: ['title', 'url', 'date', 'source']


In [52]:
#save to CSV
df.to_csv("../data/processed/cleaned_news.csv", index=False, encoding="utf-8-sig")